# Clamifision-v1 Inference Demo
Loads the fine-tuned model directly from Hugging Face -- no local files needed. Run cells top to bottom.

Model: [BJyotibrat/Clamifision-v1](https://huggingface.co/BJyotibrat/Clamifision-v1)

## 1. Install dependencies

In [1]:
!pip install -q transformers torch


## 2. Load the model from Hugging Face

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_ID = "BJyotibrat/Clamifision-v1"
MAX_LENGTH = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device)
model.eval()

print("Model loaded. Labels:", model.config.id2label)


Using device: cuda


config.json:   0%|          | 0.00/2.11k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  598MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Model loaded. Labels: {0: 'non_medical', 1: 'medical'}


## 3. Classify a single piece of text

In [3]:
def classify(text: str) -> dict:
    inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred = torch.argmax(probs).item()
    return {
        "text": text,
        "label": model.config.id2label[pred],
        "confidence": probs[pred].item(),
    }

result = classify("I've had a headache for 3 days and my vision is blurry")
print(result)


{'text': "I've had a headache for 3 days and my vision is blurry", 'label': 'medical', 'confidence': 0.9999682903289795}


## 4. Classify a batch of texts
Runs several texts at once and displays a results table. Edit `texts_to_classify` with your own text.

In [4]:
import pandas as pd

texts_to_classify = [
    "I've had a headache for 3 days and my vision is blurry",
    "whats the best pizza topping combo",
    "my chest hurts when i breathe in deeply, should i worry",
    "how do i reset my wifi router",
    "took 2 tylenol but fever wont go down, is that normal",
    "recommend me a good sci fi movie for tonight",
]

def classify_batch(texts, batch_size=16):
    all_results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)

        for j, text in enumerate(batch):
            pred = torch.argmax(probs[j]).item()
            all_results.append({
                "text": text,
                "predicted_label": model.config.id2label[pred],
                "confidence": f"{probs[j][pred].item():.4f}",
            })
    return pd.DataFrame(all_results)

results_df = classify_batch(texts_to_classify)
display(results_df)


,text,predicted_label,confidence
0,I've had a headache for 3 days and my vision i...,medical,1.0000
1,whats the best pizza topping combo,non_medical,0.9998
2,"my chest hurts when i breathe in deeply, shoul...",medical,1.0000
3,how do i reset my wifi router,non_medical,0.5230
4,"took 2 tylenol but fever wont go down, is that...",medical,1.0000
5,recommend me a good sci fi movie for tonight,non_medical,1.0000
